In [1]:
import os
from pathlib import Path
os.chdir(path = Path(r"C:\Users\apaks\projects\YT-RAG"))

In [2]:
from src.yt_rag.components.data_loader import DataLoader
from src.yt_rag.components.embedding import EmbeddingManager
from src.yt_rag.components.vectorstore import FaissVectorStore, VectorStoreManager
from src.yt_rag.components.search import RAGSearch
import time
from dotenv import load_dotenv

load_dotenv()

c:\Users\apaks\projects\YT-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "YouTube-RAG"

In [4]:
def convert_to_seconds(timestamp:str) -> int:
    """Converts timestamp (eg. '00:02:30') from str into seconds in int"""
    timestamps = timestamp.lstrip("(").rstrip(")").split(":")
    seconds = 0
    for i, timestamp in enumerate(timestamps):
        if i == 0:      # hour hand
            seconds += int(timestamp) * 60 * 60    
        if i == 1:      # minute hand
            seconds += int(timestamp) * 60 
        if i == 2:      # seconds hand
            seconds += int(timestamp) * 1
    return seconds

Create eval dataset

In [5]:
from langsmith import Client

client = Client()

dataset = client.create_dataset("RAG-eval-set-2")

In [6]:
import json
from pathlib import Path

file_path = Path("data/evals/eval_set_2.json")
with open(file_path, "rb") as f:
    data = json.load(f)

examples = data
examples[0]

{'question': 'What are the four essential characteristics of intelligent systems according to Yann LeCun?',
 'answer': 'Four essential characteristics: Capacity to understand the physical world, persistent memory, ability to reason, and ability to plan',
 'timestamp': '(0:02:49)'}

In [7]:
for example in examples:
    client.create_example(
        inputs = {"question": example["question"]},
        outputs= {"answer": example['answer']},
        metadata= {"timestamp": convert_to_seconds(example['timestamp'])},
        dataset_id= dataset.id
    )

## Evaluators

Correctness Grade

In [8]:
from typing_extensions import Annotated, TypedDict

class CorrectnessGrade(TypedDict):
    grade: Annotated[int, "Grade from 1-10"]
    correct: Annotated[bool, "True if the answer is true otherwise False"]
    explaination : Annotated[str, "Explain the reasoning for the score"]

# correctness prompt
correctness_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and a STUDENT ANSWER.

Your job is to evaluate the student answer in two ways:
1. Assign a SCORE from 1 to 10.
2. Assign a CORRECTNESS value of True or False.


- SCORE 1 means the answer is very poor: mostly or entirely incorrect, unrelated to the ground truth, or highly misleading.
- SCORE 10 means the answer is excellent: fully correct, closely aligned with the ground truth, and clearly expressed.

Here is the grading criteria to follow:

1) Grade the student answer based ONLY on its factual accuracy relative to the ground truth answer.
2) Ensure that the student answer does not contain any conflicting or contradictory statements compared to the ground truth.
3) It is OK if the student answer contains more information than the ground truth answer, as long as all additional information is factually accurate and consistent with the ground truth.
4) If the student answer includes partially correct information but also some incorrect or conflicting statements, deduct marks accordingly. The more serious or numerous the conflicts, the lower the score should be.
5) If the student answer is vague, incomplete, or only loosely related to the ground truth, give a mid-to-low score depending on how much correct information it contains.

Scoring guidelines:
- 9–10: Fully correct, no conflicts, closely matches or appropriately extends the ground truth.
- 7–8: Mostly correct, minor omissions or minor issues, no serious conflicts.
- 5–6: Partially correct, noticeable gaps or mild conflicts, but still shows some understanding.
- 3–4: Mostly incorrect or poorly aligned, with limited correct information.
- 1–2: Very poor, largely or entirely incorrect, unrelated, or highly conflicting with the ground truth.

Output format:
Provide your assement in the form of JSON object with following fields:
{
    "grade": <integer between 1 to 10>,
    "correct": <True or False>,
    "explaination": "<Explain your reasoning step by step manner, referencing specific parts of the student answer and the ground truth.>"
}

Avoid simply stating the correct answer at the outset. Focus on comparing the student answer to the ground truth and justifying the score.
""" 

In [9]:
from langchain_openai import ChatOpenAI

grader_llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0).with_structured_output(CorrectnessGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

# evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs:dict) -> dict:
    answers = f"""
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""
    result = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])

    return {
        "key": "correctness",
        "score": result["grade"],
        "value": result["correct"],
        "comment": result["explaination"]
    }   

Relevance: Response vs input

In [10]:
class RelevanceGrade(TypedDict):
    grade: Annotated[int, "Grade from 1-10"]
    relevant: Annotated[bool, "True if the answer addresses the question. False if the answer fails to address the question"]
    explaination : Annotated[str, "Explain the reasoning for the score"]

relevance_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Your job is to evaluate the student answer in two ways:
1. Assign a SCORE from 1 to 10 for relevance.
2. Assign a RELEVANCE value of True or False.

- SCORE 1 means the answer is very poor: mostly or entirely irrelevant, off-topic, or unhelpful for answering the question.
- SCORE 10 means the answer is excellent: highly relevant, directly addresses the question, and is clearly expressed.

Here is the grading criteria to follow:

1) Evaluate how directly the STUDENT ANSWER addresses the QUESTION.
2) Ensure the STUDENT ANSWER is concise and focused on the QUESTION, without unnecessary digressions.
3) The STUDENT ANSWER should meaningfully help to answer the QUESTION (not just restate it or talk around it).
4) If the STUDENT ANSWER is partially relevant but includes off-topic or distracting content, deduct marks accordingly.
5) If the STUDENT ANSWER is vague, generic, or only loosely connected to the QUESTION, give a mid-to-low score depending on how much it actually helps answer the QUESTION.

Scoring guidelines:
- 9–10: Highly relevant, directly answers the question, clear and focused.
- 7–8: Mostly relevant, minor digressions or slight lack of focus, but still clearly helps answer the question.
- 5–6: Partially relevant, noticeable vagueness or off-topic content, but some helpful information.
- 3–4: Mostly irrelevant or unhelpful, with limited connection to the question.
- 1–2: Very poor, largely or entirely irrelevant, off-topic, or confusing.

Relevance:
- Relevance = True if the answer is substantially relevant and clearly helps answer the QUESTION (typically scoring 7 or above).
- Relevance = False if the answer is mostly irrelevant, unhelpful, or only weakly connected to the QUESTION (typically scoring 6 or below).

Output format:
Provide your assessment in the form of a JSON object with the following fields:

{
  "grade": <integer between 1 and 10>,
  "relevant": <True or False>,
  "explaination": "<Explain your reasoning step by step, referencing specific parts of the student answer and the question.>"
}

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset. Focus on how well the student answer addresses the QUESTION and justifying the score.
"""

In [11]:
relevance_llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0).with_structured_output(RelevanceGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> dict:
    answer = f"""
QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['answer']}
    """

    result = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])

    return {
        "key": "relevance",
        "score": result["grade"],
        "value": result["relevant"],
        "comment": result["explaination"]
    } 

Groundedness: Response vs retrived docs

In [12]:
class GroundedGrade(TypedDict):
    grade: Annotated[int, "Grade from 1-10"]
    grounded: Annotated[bool, "True if the answer does not halucinates from the documents. False if the answer is made up"]
    explaination : Annotated[str, "Explain the reasoning for the score"]

grounded_instructions = """
You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Your job is to evaluate the student answer in two ways:
1. Assign a SCORE from 1 to 10 for groundedness.
2. Assign a GROUNDED value of True or False.

- SCORE 1 means the answer is very poor: mostly or entirely unsupported by the FACTS, highly hallucinated, or misleading.
- SCORE 10 means the answer is excellent: fully supported by the FACTS, with no hallucinated information, and clearly expressed.

Here is the grading criteria to follow:

1) Evaluate whether the STUDENT ANSWER is fully grounded in the FACTS provided.
2) The STUDENT ANSWER should not introduce any information that is not supported by, implied by, or consistent with the FACTS.
3) Additional details are acceptable only if they are clearly supported by the FACTS and do not contradict them.
4) If the STUDENT ANSWER contains partially grounded information but also some unsupported or hallucinated claims, deduct marks accordingly. The more serious or numerous the unsupported claims, the lower the score should be.
5) If the STUDENT ANSWER is vague, speculative, or goes beyond the FACTS in a way that cannot be justified by them, give a mid-to-low score depending on how much of the answer is actually grounded.

Scoring guidelines:
- 9–10: Fully grounded, no hallucinations, all claims supported by or clearly implied by the FACTS.
- 7–8: Mostly grounded, minor speculative or unclear elements, but no serious unsupported claims.
- 5–6: Partially grounded, noticeable unsupported or speculative content, but some alignment with the FACTS.
- 3–4: Mostly ungrounded, with limited connection to the FACTS and several unsupported claims.
- 1–2: Very poor, largely or entirely hallucinated, unsupported by the FACTS, or contradictory to them.

Grounded:
- Grounded = True if the answer is substantially supported by the FACTS, with no major hallucinations (typically scoring 7 or above).
- Grounded = False if the answer contains significant hallucinated, unsupported, or contradictory information (typically scoring 6 or below).

Output format:
Provide your assessment in the form of a JSON object with the following fields:

{
  "grade": <integer between 1 and 10>,
  "grounded": <True or False>,
  "explaination": "<Explain your reasoning step by step, referencing specific parts of the student answer and the FACTS.>"
}

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset. Focus on how well the student answer is grounded in the FACTS and justifying the score.

"""

In [13]:
grounded_llm = ChatOpenAI(model = "gpt-4o-mini", temperature =0).with_structured_output(GroundedGrade, 
                                                                                      method = 'json_schema', 
                                                                                      strict = True)

def groundedness(inputs: dict, outputs: dict) -> dict:
    retrived_text = "\n\n".join(outputs['relevant_chunks'])
    answer = f"FATCS:{retrived_text}\nSTUDENT ANSWER: {outputs['answer']}"

    result = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])

    return {
        "key": "groundedness",
        "score": result["grade"],
        "value": result["grounded"],
        "comment": result["explaination"]
    } 

Retrieval Relevance: Retrieved docs vs input

In [14]:
class RetrievalRelevanceGrade(TypedDict):
    grade: Annotated[int, "Grade from 1-10"]
    relevant: Annotated[bool, "True if the retrieved documents are relevant to the question, False otherwise"]
    explaination : Annotated[str, "Explain the reasoning for the score"]

retrieval_relevance_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS (retrieved documents) provided by the student.

Your job is to evaluate the relevance of the FACTS to the QUESTION in two ways:
1. Assign a SCORE from 1 to 10 for relevance.
2. Assign a RELEVANCE value of True or False.

- SCORE 1 means the FACTS are very poor: completely or almost completely unrelated to the QUESTION.
- SCORE 10 means the FACTS are excellent: clearly and strongly related to the QUESTION, with substantial semantic overlap.

Here is the grading criteria to follow:

1) Your goal is to identify whether the FACTS are related to the QUESTION.
2) If the FACTS contain ANY keywords, concepts, or semantic meaning related to the QUESTION, consider them relevant to some degree.
3) It is OK if the FACTS contain SOME information that is unrelated to the QUESTION, as long as they also contain information that is clearly related.
4) If only a small portion of the FACTS is related and most of the content is off-topic, give a mid-to-low score depending on how much relevant information is present.
5) If the FACTS are entirely off-topic, generic, or about a different subject, give a very low score.

Scoring guidelines:
- 9–10: Highly relevant, strong semantic overlap with the QUESTION, clearly useful for answering it.
- 7–8: Mostly relevant, good overlap with the QUESTION, some off-topic content but still clearly useful.
- 5–6: Partially relevant, noticeable off-topic content, but some meaningful connection to the QUESTION.
- 3–4: Weakly relevant, only minor or superficial connection to the QUESTION.
- 1–2: Very poor, largely or completely unrelated to the QUESTION.

Relevance:
- Relevance = True if the FACTS contain ANY meaningful keywords, concepts, or semantic content related to the QUESTION (typically scoring 5 or above).
- Relevance = False if the FACTS are completely or almost completely unrelated to the QUESTION (typically scoring 4 or below).

Output format:
Provide your assessment in the form of a JSON object with the following fields:

{
  "grade": <integer between 1 and 10>,
  "relevant": <True or False>,
  "explaination": "<Explain your reasoning step by step, referencing specific parts of the FACTS and the QUESTION.>"
}

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset. Focus on how well the FACTS relate to the QUESTION and justifying the score.
"""

In [15]:
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> dict:
    """An evaluator for document relevance"""
    retrived_text = "\n\n".join(outputs['relevant_chunks'])
    answer = f"FACTS: {retrived_text}\nQUESTION: {inputs['question']}"

    # Run evaluator
    result = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return {
        "key": "retrieval_relevance",
        "score": result["grade"],
        "value": result["relevant"],
        "comment": result["explaination"]
    } 

Timestamp Error
- How far off retrieval is from the true timestamp of the answer. 
- Will only consider the timestamp of the most relevant chunk

In [16]:
def timestamp_error(outputs: dict, reference_outputs:dict) -> dict:
    correct_ts = convert_to_seconds(reference_outputs['timestamp'])
    retrieved_ts = outputs['timestamp']

    if retrieved_ts is None:
        return {
            "key": "timestamp_error",
            "score": None,
            "value": None,
            "comment": "No timestamp was extracted from the retrieved document."
        }

    error = abs(correct_ts - retrieved_ts)

    return {
        "key": "timestamp_error",
        "score": error,
        "value": error,  # optional, but allowed
        "comment": f"The retrieved document timestamp ({retrieved_ts}s) differs from the correct timestamp ({correct_ts}s) by {error} seconds."
    }

### Run Evals

In [17]:
from langsmith import traceable

# define the function to run the full rag pipeline
@traceable()
def run_rag_pipeline(url, query):
    rag = RAGSearch(url = url)
    start = time.time()
    relevant_chunks = rag.search(query= query, top_k = 5)
    context = " ".join(relevant_chunks)
    response = rag.generate_response(context=context, query=query)
    timestamps = rag.get_video_timestamps()
    if timestamps and isinstance(timestamps[0], (list, tuple)) and len(timestamps[0]) >= 1:
        timestamp = timestamps[0][0]
    else:
        timestamp = None

    end = time.time()

    return {
        "answer": response,
        "relevant_chunks": relevant_chunks,
        "timestamp": timestamp,               # now correct
        "runtime_duration": end - start
    }

In [18]:
url = "https://www.youtube.com/watch?v=5t1vTLU7s40"

In [ ]:
def target(inputs:dict) -> dict:
    return run_rag_pipeline(url = url, query=inputs['question'])

experiment_results = client.evaluate(
    target,
    data = "RAG-eval-set-2",
    evaluators = [correctness, groundedness, relevance, retrieval_relevance, timestamp_error],
    experiment_prefix="rag-detail-evaluation",
    metadata={"version": "LCEL context, gpt-4-0125-preview"}
)

View the evaluation results for experiment: 'rag-detail-evaluation-0bcacca4' at:
https://smith.langchain.com/o/92a32521-1999-4625-877a-20ef1a977765/datasets/98f1e9e9-fb6f-4387-8f81-3ab7a430c12b/compare?selectedSessions=f4fa8911-6ed3-4607-85d4-d49ae9623f00




20it [04:02, 12.11s/it]


In [20]:
eval_results = experiment_results.to_pandas()
eval_results

,inputs.question,outputs.answer,outputs.relevant_chunks,outputs.timestamp,outputs.runtime_duration,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,Why does LeCun argue that AGI will not be an '...,Final Answer: LeCun argues that AGI will not b...,"[I mean, there's a bit coming from Silicon tec...",7383.030,4.018230,None,AGI is a long-term goal that will be reached t...,9,9,9,9,4.032188,a507a029-fb0d-4247-9c2a-94acad687c47,019fd474-1c6f-7d53-9587-767d2ab6859b
1,What is the future outlook for Llama developme...,Final Answer: The future outlook for LLaMA dev...,"[that are really experienced building it, you'...",7028.220,3.552684,None,"Iterative improvements, better multimodal capa...",9,9,9,9,3.558569,92e52337-4e2d-4044-a238-8af451f6c499,019fd474-4d80-75f3-b922-e6634572d07b
2,Why does LeCun compare the potential of AI to ...,The retrieved context does not contain enough ...,"[No, it took decades to kind of fine tune the ...",8079.030,2.042047,None,Instructions don't help build a physical rocke...,1,1,1,5,2.061476,cb0dd5c3-8384-40e4-b17d-fb88e4dc9851,019fd474-88be-7c31-bcdb-6d8f15188530
3,What is the limitation of relying on language ...,Final Answer: The limitation of relying on lan...,[filter the data very quickly. Somebody might ...,376.620,3.679489,None,Language-based systems cannot capture the high...,8,9,9,9,3.689762,13c9419c-8e2d-48c4-b0af-cbba70dd97b1,019fd474-ab5d-7840-b524-0517dedc789d
4,What is the concern regarding AI models becomi...,Final Answer: The concern regarding AI models ...,"[So you can ask humans to rate answers, multip...",5576.505,3.195011,None,The risk that over-tuning for specific social ...,8,9,9,9,3.222337,5c8210d4-183d-479d-91cc-433a5a8fd2ca,019fd474-de14-7ca3-814c-0284e9b3780a
5,How does LeCun view the role of reinforcement ...,Final Answer: LeCun views the role of reinforc...,[was gonna do something and they do something ...,5463.330,2.925470,None,"It can be used to fine-tune systems, but LeCun...",7,9,9,8,2.943792,861f1a00-c220-4f8d-86c6-1d893f5216cd,019fd475-0ade-7503-b2c8-a7dd915372d3
6,What is the significance of the Marvack Parado...,Final Answer: The significance of Moravec's pa...,[- I don't think it's just Moravec's paradox. ...,7618.702,3.515151,None,It highlights that high-level reasoning is oft...,8,9,9,9,3.526489,1e39bfb1-7335-4131-bddd-4c67d8a11b45,019fd475-4186-7f91-bc0a-3609d0fb4f70
7,How does Meta's business model accommodate ope...,Final Answer: Meta's business model accommodat...,[if you have a big enough potential customer b...,6334.680,3.194655,None,It leverages a large existing user/customer ba...,8,9,9,9,3.214055,9e1270ad-3abb-4a59-9f0c-084e0cb19b4a,019fd475-711a-7d71-9ba4-aecb04040d70
8,Why does LeCun advocate for open source AI pla...,Final Answer: LeCun advocates for open source ...,[because those systems will constitute the rep...,5958.923,3.024749,None,To ensure diverse AI systems that represent va...,10,9,10,10,3.044661,005e3dea-73ac-42bc-9955-b1ef1ad2b79a,019fd475-9e37-71a3-80a1-fcf06932f5a6
9,How does LeCun define the concept of 'energy' ...,"Final Answer: In an energy-based model, LeCun ...",[How do you know what is an answer that's bett...,4918.093,3.422897,None,"In an energy-based model, energy is a scalar (...",7,9,9,9,3.442099,e6fc7626-dd88-45c8-87ab-25ddf408d477,019fd475-cb43-7f41-a406-5c90fa9d0ec6


In [21]:
import pandas as pd
from datetime import datetime

def save_eval_results(obj:pd.DataFrame, dir_path:str = "eval_results"):
    """Save Evaluation results as excel"""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    dir = Path(dir_path)
    dir.mkdir(exist_ok=True)
    file_path = dir / f"eval_{timestamp}.csv"
    obj.to_csv(file_path)

save_eval_results(obj=eval_results)